# 03. Vectorization & Universal Functions (ufuncs): Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **03. Vectorization & Universal Functions (ufuncs)**. Universal functions (`ufuncs`) are C-implemented compiled routines that operate element-by-element across `ndarray` buffers with hardware SIMD acceleration. This notebook explores core mathematical ufuncs (`sqrt`, `exp`, `log`, `sin`, `cos`, `abs`, `clip`, `sign`, `round`), ufunc reduction and accumulation methods (`reduce`, `accumulate`, `outer`), unbuffered in-place updates (`ufunc.at`), and custom vectorized function compilation (`np.vectorize`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Arithmetic Vectorization (`+`, `-`, `*`, `/`)
- [x] 🔹 Square Root: `np.sqrt()`
- [x] 🔹 Exponential: `np.exp()`
- [x] 🔹 Natural Logarithm: `np.log()`
- [x] 🔹 Trigonometric Functions: `np.sin()` & `np.cos()`
- [x] 🔹 Absolute Value: `np.abs()`
- [x] 🔹 Cumulative Reductions with `ufunc.reduce()`
- [x] 🔹 Running Totals with `ufunc.accumulate()`
- [x] 🔹 Outer Products with `ufunc.outer()`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Arithmetic Vectorization (`+`, `-`, `*`, `/`)
- **What it does:** Applies fee multipliers and taxes element-wise to numerical values.
- **Syntax:** `+`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Arithmetic Vectorization (`+`, `-`, `*`, `/`) across the extracted numeric transaction `amounts` array to compute performance metrics.


In [2]:
print('Amounts with 5% Fee Applied (first 5):', (amounts[:5] * 1.05).round(2))

Amounts with 5% Fee Applied (first 5): [ 638.17 1910.07   67.28 1077.02  811.38]


### 🔹 Square Root: `np.sqrt()`
- **What it does:** Computes square root of numerical values for variance scaling.
- **Syntax:** `np.sqrt()`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Square Root across the extracted numeric transaction `amounts` array to compute performance metrics.


In [3]:
print('Square Roots of Amounts:', np.sqrt(amounts[:5]).round(2))

Square Roots of Amounts: [24.65 42.65  8.   32.03 27.8 ]


### 🔹 Exponential: `np.exp()`
- **What it does:** Computes exponential scaling for risk logits.
- **Syntax:** `np.exp()`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Exponential with practical fintech data structures and variables in the following code block.


In [4]:
print('Exponential Account Age Factors:', np.exp(account_ages[:5] / 12.0).round(2))

Exponential Account Age Factors: [1.95000e+00 1.03100e+01 1.96517e+03 6.45000e+01 1.52000e+00]


### 🔹 Natural Logarithm: `np.log()`
- **What it does:** Applies log-transformation $\ln(1 + x)$ to normalize skewed numerical values.
- **Syntax:** `np.log()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Natural Logarithm across the extracted numeric transaction `amounts` array to compute performance metrics.


In [5]:
log_amounts = np.log1p(amounts[:5])
print('Log-Transformed Amounts (ln(1+x)):', log_amounts.round(3))

Log-Transformed Amounts (ln(1+x)): [6.411 7.507 4.176 6.934 6.651]


### 🔹 Trigonometric Functions: `np.sin()` & `np.cos()`
- **What it does:** Computes cyclical trigonometric time encodings.
- **Syntax:** `np.sin()`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Trigonometric Functions across the extracted numeric transaction `amounts` array to compute performance metrics.


In [6]:
cyclical_feature = np.sin(2 * np.pi * (amounts[:5] % 365) / 365)
print('Cyclical Encodings:', cyclical_feature.round(3))

Cyclical Encodings: [-0.861 -0.101  0.893 -0.929  0.671]


### 🔹 Absolute Value: `np.abs()`
- **What it does:** Computes absolute deviation from median numerical values.
- **Syntax:** `np.abs()`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Absolute Value across the extracted numeric transaction `amounts` array to compute performance metrics.


In [7]:
abs_devs = np.abs(amounts[:5] - np.median(amounts))
print('Absolute Deviations from Median:', abs_devs.round(2))

Absolute Deviations from Median: [390.5  820.83 934.2   27.45 225.54]


### 🔹 Cumulative Reductions with `ufunc.reduce()`
- **What it does:** Calculates total revenue using `np.add.reduce`.
- **Syntax:** `ufunc.reduce()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Cumulative Reductions with `ufunc.reduce()` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [8]:
total_revenue = np.add.reduce(amounts)
print(f'Total Transaction Volume (np.add.reduce): ${total_revenue:,.2f}')

Total Transaction Volume (np.add.reduce): $14,326,935.50


### 🔹 Running Totals with `ufunc.accumulate()`
- **What it does:** Calculates running cumulative revenue stream across elements.
- **Syntax:** `ufunc.accumulate()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Running Totals with `ufunc.accumulate()` across the extracted numeric transaction `amounts` array to compute performance metrics.


In [9]:
running_revenue = np.add.accumulate(amounts[:5])
print('Running Cumulative Revenue (first 5):', running_revenue.round(2))

Running Cumulative Revenue (first 5): [ 607.78 2426.89 2490.97 3516.7  4289.44]


### 🔹 Outer Products with `ufunc.outer()`
- **What it does:** Computes outer cross-product risk matrix between card fee tiers and account ages.
- **Syntax:** `ufunc.outer()`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Outer Products with `ufunc.outer()` with practical fintech data structures and variables in the following code block.


In [10]:
fees = np.array([0.015, 0.025, 0.035])
outer_fee_matrix = np.multiply.outer(fees, account_ages[:4])
print('Outer Fee Scaling Matrix (3 fees x 4 accounts):\n', outer_fee_matrix.round(2))

Outer Fee Scaling Matrix (3 fees x 4 accounts):
 [[0.12 0.42 1.36 0.75]
 [0.2  0.7  2.28 1.25]
 [0.28 0.98 3.19 1.75]]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Pairwise Transaction Distance Matrix via `np.subtract.outer`
- **Objective:** Q1: Pairwise Transaction Distance Matrix via `np.subtract.outer`
- **Approach:** Compute the complete pairwise absolute difference matrix for the first 5 transaction amounts without Python loops.
- **Syntax:** `np.abs(np.subtract.outer(amounts[:5], amounts[:5]))`

In [11]:
pw_matrix = np.abs(np.subtract.outer(amounts[:5], amounts[:5]))
print('Pairwise Amount Differences Matrix:\n', pw_matrix.round(2))

Pairwise Amount Differences Matrix:
 [[   0.   1211.33  543.7   417.95  164.96]
 [1211.33    0.   1755.03  793.38 1046.37]
 [ 543.7  1755.03    0.    961.65  708.66]
 [ 417.95  793.38  961.65    0.    252.99]
 [ 164.96 1046.37  708.66  252.99    0.  ]]
